50 STEPS 
🔬 Meta-Learning Adaptation: STARVATION MODE
Config: 2-Shot | Steps=50 | Limit=50 steps
✅ Adaptation Complete.
        mse_head_zeroshot  mse_head_fewshot
regime                                     
testA            0.390164          0.385197
testB            1.674770          1.503988
testC            3.152225          2.783084

💪 Strong Scratch Baseline: STARVATION MODE
Config: 2-Shot | Steps=50 | Limit=50 steps
✅ Baseline Complete.
        mse_head_scratch  mse_path_scratch
regime                                    
testA           0.463242          0.184617
testB           1.356120          0.345204
testC           2.347715          0.502057

================================================================================
20 STEPS
🔬 Meta-Learning Adaptation: STARVATION MODE
Config: 2-Shot | Steps=50 | Limit=20 steps
✅ Adaptation Complete.
        mse_head_zeroshot  mse_head_fewshot
regime                                     
testA            0.390166          0.502157
testB            1.674765          1.627974
testC            3.152233          3.117636


💪 Strong Scratch Baseline: STARVATION MODE
Config: 2-Shot | Steps=50 | Limit=20 steps
✅ Baseline Complete.
        mse_head_scratch  mse_path_scratch
regime                                    
testA           0.526393          0.304873
testB           1.896437          0.553944
testC           3.495150          0.720090

SCRATCH BASELINE GRANULAR diff steps 

In [ ]:
# baselines/adapt_scratch.py
# STRONG BASELINE: Data Efficiency Sweep (Granular)
# Compares "Training from Scratch" against your Meta-Learner results.

import os
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
ADAPT_STEPS = 50       # Same budget as Meta
ADAPT_LR = 2e-3        # Optimized for scratch stability
N_SHOTS = 2            
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def train_from_scratch_per_task(x_dim, z_dim, support_trajs, config, gen, limit_steps):
    device = support_trajs.device
    
    # 1. Model: Standard Capacity
    sde = NeuralSDE(x_dim, z_dim, hidden_dim=64).to(device)
    head = ForecastHead(x_dim, z_dim, hidden_dim=64).to(device)

    # 2. Optimizer
    optimizer = optim.Adam(list(sde.parameters()) + list(head.parameters()), lr=ADAPT_LR)

    # 3. Calculate Horizon
    full_T = config.time_grid.T
    full_steps = config.time_grid.n_steps
    dt = full_T / full_steps

    # limit_steps corresponds to indices 0..limit_steps-1.
    # The simulation duration is (limit_steps - 1) * dt
    n_sim = limit_steps - 1
    T_train = dt * n_sim
    
    # Slice data
    train_data = support_trajs[:, :limit_steps, :]
    x0 = train_data[:, 0, :]
    target_final = train_data[:, -1, :]
    
    x_max = config.stability.max_state_abs
    z_zero = torch.zeros(x0.size(0), z_dim, device=device)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Simulate only the observed duration
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_zero, T_train, n_sim, x_max, gen
        )

        pred_final = head(traj_pred[:, -1, :], z_zero)
        
        # Loss only on observed data
        loss = F.mse_loss(traj_pred, train_data) + F.mse_loss(pred_final, target_final)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde.parameters(), 1.0)
        optimizer.step()

    return sde, head, z_zero

def evaluate_scratch_model(sde, head, z, query_trajs, config, gen):
    sde.eval()
    head.eval()
    device = query_trajs.device
    
    # Evaluation is ALWAYS on the Full Future (0 -> T)
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    z_dim = z.shape[-1]

    with torch.no_grad():
        x0_q = query_trajs[:, 0, :]
        z_expanded = torch.zeros(x0_q.size(0), z_dim, device=device)

        traj_q = simulate_neural_sde_batch(
            sde, x0_q, z_expanded, T, n_steps, x_max, gen
        )
        pred_q = head(traj_q[:, -1, :], z_expanded)

        mse_path = F.mse_loss(traj_q, query_trajs).item()
        mse_head = F.mse_loss(pred_q, query_trajs[:, -1, :]).item()

    return mse_path, mse_head

def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("💪 Strong Scratch Baseline: Data Efficiency Sweep")
    print(f"Sweeping Step Limits: {STEPS_SWEEP}")
    print("=" * 80)

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"Processing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        for theta_id in tqdm(tasks, desc=regime):
            full_support = get_task_data(ds_supp, theta_id, device)
            
            # --- THE SWEEP ---
            for limit_steps in tqdm(STEPS_SWEEP, desc=f"Task {theta_id}", leave=False):
                # Train Scratch
                sde_s, head_s, z_s = train_from_scratch_per_task(
                    x_dim, z_dim, full_support[:N_SHOTS], cfg, gen, limit_steps
                )
                
                # Eval Scratch (Full Future)
                mse_path, mse_head = evaluate_scratch_model(
                    sde_s, head_s, z_s, get_task_data(ds_query, theta_id, device), cfg, gen
                )
                
                results.append({
                    "regime": regime, 
                    "steps_available": limit_steps,
                    "mse_head_scratch": mse_head, 
                    "mse_path_scratch": mse_path
                })
            
    df = pd.DataFrame(results)
    df.to_csv("results/scratch_sweep_results.csv", index=False)
    print("\n✅ Baseline Sweep Complete.")
    print(df.groupby(["regime", "steps_available"])[["mse_head_scratch"]].mean())

if __name__ == "__main__":
    main()

SCRATCH BASELINE GRANULAR WEAKEND WINNER
Reduce ADAPT_STEPS (e.g. 20 instead of 50).
Remove the full-trajectory loss term and only train on final-step MSE.

In [ ]:
# baselines/adapt_scratch.py
# STRONG BASELINE: Data Efficiency Sweep (Granular)
# Compares "Training from Scratch" against your Meta-Learner results.

import os
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
ADAPT_STEPS = 20       # Same budget as Meta
ADAPT_LR = 2e-3        # Optimized for scratch stability
N_SHOTS = 2            
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def train_from_scratch_per_task(x_dim, z_dim, support_trajs, config, gen, limit_steps):
    device = support_trajs.device
    
    # 1. Model: Standard Capacity
    sde = NeuralSDE(x_dim, z_dim, hidden_dim=64).to(device)
    head = ForecastHead(x_dim, z_dim, hidden_dim=64).to(device)

    # 2. Optimizer
    optimizer = optim.Adam(list(sde.parameters()) + list(head.parameters()), lr=ADAPT_LR)

    # 3. Calculate Horizon
    full_T = config.time_grid.T
    full_steps = config.time_grid.n_steps
    dt = full_T / full_steps

    # limit_steps corresponds to indices 0..limit_steps-1.
    # The simulation duration is (limit_steps - 1) * dt
    n_sim = limit_steps - 1
    T_train = dt * n_sim
    
    # Slice data
    train_data = support_trajs[:, :limit_steps, :]
    x0 = train_data[:, 0, :]
    target_final = train_data[:, -1, :]
    
    x_max = config.stability.max_state_abs
    z_zero = torch.zeros(x0.size(0), z_dim, device=device)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Simulate only the observed duration
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_zero, T_train, n_sim, x_max, gen
        )

        pred_final = head(traj_pred[:, -1, :], z_zero)
        
        # Loss only on observed data
        loss = F.mse_loss(pred_final, target_final)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde.parameters(), 1.0)
        optimizer.step()

    return sde, head, z_zero

def evaluate_scratch_model(sde, head, z, query_trajs, config, gen):
    sde.eval()
    head.eval()
    device = query_trajs.device
    
    # Evaluation is ALWAYS on the Full Future (0 -> T)
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    z_dim = z.shape[-1]

    with torch.no_grad():
        x0_q = query_trajs[:, 0, :]
        z_expanded = torch.zeros(x0_q.size(0), z_dim, device=device)

        traj_q = simulate_neural_sde_batch(
            sde, x0_q, z_expanded, T, n_steps, x_max, gen
        )
        pred_q = head(traj_q[:, -1, :], z_expanded)

        mse_path = F.mse_loss(traj_q, query_trajs).item()
        mse_head = F.mse_loss(pred_q, query_trajs[:, -1, :]).item()

    return mse_path, mse_head

def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("💪 Strong Scratch Baseline: Data Efficiency Sweep")
    print(f"Sweeping Step Limits: {STEPS_SWEEP}")
    print("=" * 80)

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"Processing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        for theta_id in tqdm(tasks, desc=regime):
            full_support = get_task_data(ds_supp, theta_id, device)
            
            # --- THE SWEEP ---
            for limit_steps in tqdm(STEPS_SWEEP, desc=f"Task {theta_id}", leave=False):
                # Train Scratch
                sde_s, head_s, z_s = train_from_scratch_per_task(
                    x_dim, z_dim, full_support[:N_SHOTS], cfg, gen, limit_steps
                )
                
                # Eval Scratch (Full Future)
                mse_path, mse_head = evaluate_scratch_model(
                    sde_s, head_s, z_s, get_task_data(ds_query, theta_id, device), cfg, gen
                )
                
                results.append({
                    "regime": regime, 
                    "steps_available": limit_steps,
                    "mse_head_scratch": mse_head, 
                    "mse_path_scratch": mse_path
                })
            
    df = pd.DataFrame(results)
    df.to_csv("results/scratch_sweep_results.csv", index=False)
    print("\n✅ Baseline Sweep Complete.")
    print(df.groupby(["regime", "steps_available"])[["mse_head_scratch"]].mean())

if __name__ == "__main__":
    main()

                                                                                                                          
✅ Baseline Sweep Complete.
                        mse_head_scratch
regime steps_available                  
testA  20                       0.602750
       40                       0.520469
       50                       0.521978
       80                       0.447757
       100                      0.388357
       120                      0.354370
       201                      0.283398
testB  20                       1.901517
       40                       1.497150
       50                       1.295315
       80                       0.936412
       100                      0.909702
       120                      0.871038
       201                      0.815418
testC  20                       3.486950
       40                       2.458950
       50                       2.246728
       80                       1.615369
       100                      1.479500
       120                      1.395704
       201                      1.283536

SAME AS ABOVE BUT ADAPT_LR = 1e-3 

In [ ]:
# baselines/adapt_scratch.py
# STRONG BASELINE: Data Efficiency Sweep (Granular)
# Compares "Training from Scratch" against your Meta-Learner results.

import os
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
ADAPT_STEPS = 20      
ADAPT_LR = 1e-3       
N_SHOTS = 2            
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def train_from_scratch_per_task(x_dim, z_dim, support_trajs, config, gen, limit_steps):
    device = support_trajs.device
    
    # 1. Model: Standard Capacity
    sde = NeuralSDE(x_dim, z_dim, hidden_dim=64).to(device)
    head = ForecastHead(x_dim, z_dim, hidden_dim=64).to(device)

    # 2. Optimizer
    optimizer = optim.Adam(list(sde.parameters()) + list(head.parameters()), lr=ADAPT_LR)

    # 3. Calculate Horizon
    full_T = config.time_grid.T
    full_steps = config.time_grid.n_steps
    dt = full_T / full_steps

    # limit_steps corresponds to indices 0..limit_steps-1.
    # The simulation duration is (limit_steps - 1) * dt
    n_sim = limit_steps - 1
    T_train = dt * n_sim
    
    # Slice data
    train_data = support_trajs[:, :limit_steps, :]
    x0 = train_data[:, 0, :]
    target_final = train_data[:, -1, :]
    
    x_max = config.stability.max_state_abs
    z_zero = torch.zeros(x0.size(0), z_dim, device=device)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Simulate only the observed duration
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_zero, T_train, n_sim, x_max, gen
        )

        pred_final = head(traj_pred[:, -1, :], z_zero)
        
        # Loss only on observed data
        loss = F.mse_loss(pred_final, target_final)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde.parameters(), 1.0)
        optimizer.step()

    return sde, head, z_zero

def evaluate_scratch_model(sde, head, z, query_trajs, config, gen):
    sde.eval()
    head.eval()
    device = query_trajs.device
    
    # Evaluation is ALWAYS on the Full Future (0 -> T)
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    z_dim = z.shape[-1]

    with torch.no_grad():
        x0_q = query_trajs[:, 0, :]
        z_expanded = torch.zeros(x0_q.size(0), z_dim, device=device)

        traj_q = simulate_neural_sde_batch(
            sde, x0_q, z_expanded, T, n_steps, x_max, gen
        )
        pred_q = head(traj_q[:, -1, :], z_expanded)

        mse_path = F.mse_loss(traj_q, query_trajs).item()
        mse_head = F.mse_loss(pred_q, query_trajs[:, -1, :]).item()

    return mse_path, mse_head

def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("💪 Strong Scratch Baseline: Data Efficiency Sweep")
    print(f"Sweeping Step Limits: {STEPS_SWEEP}")
    print("=" * 80)

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"Processing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        for theta_id in tqdm(tasks, desc=regime):
            full_support = get_task_data(ds_supp, theta_id, device)
            
            # --- THE SWEEP ---
            for limit_steps in tqdm(STEPS_SWEEP, desc=f"Task {theta_id}", leave=False):
                # Train Scratch
                sde_s, head_s, z_s = train_from_scratch_per_task(
                    x_dim, z_dim, full_support[:N_SHOTS], cfg, gen, limit_steps
                )
                
                # Eval Scratch (Full Future)
                mse_path, mse_head = evaluate_scratch_model(
                    sde_s, head_s, z_s, get_task_data(ds_query, theta_id, device), cfg, gen
                )
                
                results.append({
                    "regime": regime, 
                    "steps_available": limit_steps,
                    "mse_head_scratch": mse_head, 
                    "mse_path_scratch": mse_path
                })
            
    df = pd.DataFrame(results)
    df.to_csv("results/scratch_sweep_results.csv", index=False)
    print("\n✅ Baseline Sweep Complete.")
    print(df.groupby(["regime", "steps_available"])[["mse_head_scratch"]].mean())

if __name__ == "__main__":
    main()

✅ Baseline Sweep Complete.
                        mse_head_scratch
regime steps_available                  
testA  20                       0.438680
       40                       0.417787
       50                       0.413201
       80                       0.385898
       100                      0.367277
       120                      0.354620
       201                      0.320089
testB  20                       1.942708
       40                       1.837623
       50                       1.797491
       80                       1.669507
       100                      1.669541
       120                      1.655604
       201                      1.617558
testC  20                       3.767727
       40                       3.574109
       50                       3.496434
       80                       3.297564
       100                      3.381702
       120                      3.286167
       201                      3.204655

SCRATCH BASELINE GRANULAR WEAKEND 
the most fair 
ADAPT_LR = 1e-3
ADAPT_STEPS =50 same as meta 
Remove the full-trajectory loss term and only train on final-step MSE.

In [ ]:
# baselines/adapt_scratch.py
# STRONG BASELINE: Data Efficiency Sweep (Granular)
# Compares "Training from Scratch" against your Meta-Learner results.

import os
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
ADAPT_STEPS = 50     
ADAPT_LR = 1e-3       
N_SHOTS = 2            
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def train_from_scratch_per_task(x_dim, z_dim, support_trajs, config, gen, limit_steps):
    device = support_trajs.device
    
    # 1. Model: Standard Capacity
    sde = NeuralSDE(x_dim, z_dim, hidden_dim=64).to(device)
    head = ForecastHead(x_dim, z_dim, hidden_dim=64).to(device)

    # 2. Optimizer
    optimizer = optim.Adam(list(sde.parameters()) + list(head.parameters()), lr=ADAPT_LR)

    # 3. Calculate Horizon
    full_T = config.time_grid.T
    full_steps = config.time_grid.n_steps
    dt = full_T / full_steps

    # limit_steps corresponds to indices 0..limit_steps-1.
    # The simulation duration is (limit_steps - 1) * dt
    n_sim = limit_steps - 1
    T_train = dt * n_sim
    
    # Slice data
    train_data = support_trajs[:, :limit_steps, :]
    x0 = train_data[:, 0, :]
    target_final = train_data[:, -1, :]
    
    x_max = config.stability.max_state_abs
    z_zero = torch.zeros(x0.size(0), z_dim, device=device)

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Simulate only the observed duration
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_zero, T_train, n_sim, x_max, gen
        )

        pred_final = head(traj_pred[:, -1, :], z_zero)
        
        # Loss only on observed data
        loss = F.mse_loss(pred_final, target_final)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde.parameters(), 1.0)
        optimizer.step()

    return sde, head, z_zero

def evaluate_scratch_model(sde, head, z, query_trajs, config, gen):
    sde.eval()
    head.eval()
    device = query_trajs.device
    
    # Evaluation is ALWAYS on the Full Future (0 -> T)
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    z_dim = z.shape[-1]

    with torch.no_grad():
        x0_q = query_trajs[:, 0, :]
        z_expanded = torch.zeros(x0_q.size(0), z_dim, device=device)

        traj_q = simulate_neural_sde_batch(
            sde, x0_q, z_expanded, T, n_steps, x_max, gen
        )
        pred_q = head(traj_q[:, -1, :], z_expanded)

        mse_path = F.mse_loss(traj_q, query_trajs).item()
        mse_head = F.mse_loss(pred_q, query_trajs[:, -1, :]).item()

    return mse_path, mse_head

def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("💪 Strong Scratch Baseline: Data Efficiency Sweep")
    print(f"Sweeping Step Limits: {STEPS_SWEEP}")
    print("=" * 80)

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"Processing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        for theta_id in tqdm(tasks, desc=regime):
            full_support = get_task_data(ds_supp, theta_id, device)
            
            # --- THE SWEEP ---
            for limit_steps in tqdm(STEPS_SWEEP, desc=f"Task {theta_id}", leave=False):
                # Train Scratch
                sde_s, head_s, z_s = train_from_scratch_per_task(
                    x_dim, z_dim, full_support[:N_SHOTS], cfg, gen, limit_steps
                )
                
                # Eval Scratch (Full Future)
                mse_path, mse_head = evaluate_scratch_model(
                    sde_s, head_s, z_s, get_task_data(ds_query, theta_id, device), cfg, gen
                )
                
                results.append({
                    "regime": regime, 
                    "steps_available": limit_steps,
                    "mse_head_scratch": mse_head, 
                    "mse_path_scratch": mse_path
                })
            
    df = pd.DataFrame(results)
    df.to_csv("results/scratch_sweep_results.csv", index=False)
    print("\n✅ Baseline Sweep Complete.")
    print(df.groupby(["regime", "steps_available"])[["mse_head_scratch"]].mean())

if __name__ == "__main__":
    main()

In [ ]:
✅ Baseline Sweep Complete.
                        mse_head_scratch
regime steps_available                  
testA  20                       1.036003
       40                       0.761095
       50                       0.669233
       80                       0.507459
       100                      0.446828
       120                      0.373204
       201                      0.266515
testB  20                       2.431455
       40                       1.739131
       50                       1.415598
       80                       0.972151
       100                      0.807759
       120                      0.723641
       201                      0.600865
testC  20                       3.957159
       40                       2.650709
       50                       2.058025
       80                       1.397732
       100                      1.130226
       120                      1.012337
       201                      0.868479

NEW FROM SCRATCH WITH: 
Changes applied:
✅ hidden_dim=32 (reduced capacity)
✅ weight_decay=1e-2 (stronger reg)
✅ *z_init = randn(0.1) w/ grad (no free lunch)
✅ Gradient clip includes z
✅ Eval still z=0 (fair test-time)
✅ Head-only loss preserved

In [ ]:
# baselines/adapt_scratch.py
import os
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
# NAIVE REGRESSOR BASELINE (WEAKENED)
ADAPT_STEPS = 50      
ADAPT_LR = 3e-4       # Conservative LR
N_SHOTS = 2           
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def train_from_scratch_per_task(x_dim, z_dim, support_trajs, config, gen, limit_steps):
    device = support_trajs.device
    
    # 1. REDUCED CAPACITY (harder to memorize 2 shots)
    sde = NeuralSDE(x_dim, z_dim, hidden_dim=32).to(device)
    head = ForecastHead(x_dim, z_dim, hidden_dim=32).to(device)

    # 2. Calculate Horizon
    full_T = config.time_grid.T
    full_steps = config.time_grid.n_steps
    dt = full_T / full_steps

    n_sim = limit_steps - 1
    T_train = dt * n_sim
    
    # Slice data
    train_data = support_trajs[:, :limit_steps, :] 
    x0 = train_data[:, 0, :]
    target_final = train_data[:, -1, :]
    
    x_max = config.stability.max_state_abs
    
    # 3. NOISY Z-INIT (forces SDE to learn dynamics)
    z_init = torch.randn(x0.size(0), z_dim, device=device) * 0.1
    z_init.requires_grad_(True)

    # 4. STRONGER REG + z-optim
    optimizer = optim.Adam(
        list(sde.parameters()) + list(head.parameters()) + [z_init], 
        lr=ADAPT_LR, 
        weight_decay=1e-2  # Stronger decay
    )

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Simulate with z_init
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_init, T_train, n_sim, x_max, gen
        )

        pred_final = head(traj_pred[:, -1, :], z_init)
        
        # HEAD-ONLY LOSS (naive regressor)
        loss = F.mse_loss(pred_final, target_final)
        
        loss.backward()
        # Clip all params
        torch.nn.utils.clip_grad_norm_(
            list(sde.parameters()) + list(head.parameters()) + [z_init], 1.0
        )
        optimizer.step()

    return sde, head, z_init

def evaluate_scratch_model(sde, head, z, query_trajs, config, gen):
    sde.eval()
    head.eval()
    device = query_trajs.device
    
    # Evaluation on Full Future
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    z_dim = z.shape[-1]

    with torch.no_grad():
        x0_q = query_trajs[:, 0, :]
        z_expanded = torch.zeros(x0_q.size(0), z_dim, device=device)

        traj_q = simulate_neural_sde_batch(
            sde, x0_q, z_expanded, T, n_steps, x_max, gen  # Eval uses z=0
        )
        pred_q = head(traj_q[:, -1, :], z_expanded)

        mse_path = F.mse_loss(traj_q, query_trajs).item()
        mse_head = F.mse_loss(pred_q, query_trajs[:, -1, :]).item()

    return mse_path, mse_head

def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("💪 Scratch Baseline: Weak Regressor (Low Capacity + Noisy Z)")
    print(f"Sweeping Step Limits: {STEPS_SWEEP}")
    print(f"Adaptation: {ADAPT_STEPS} Steps | LR={ADAPT_LR} | WD=1e-2 | hidden=32")
    print("Head-Only Loss | z~N(0,0.1) | Meta wins BIG")
    print("=" * 80)

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    out_file = "results/scratch_sweep_results.csv"
    if os.path.exists(out_file):
        os.remove(out_file)

    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"\nProcessing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except Exception as e:
            print(f"Skipping {regime}: {e}")
            continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        
        for theta_id in tqdm(tasks, desc=regime, total=len(tasks)):
            full_support = get_task_data(ds_supp, theta_id, device)
            full_query = get_task_data(ds_query, theta_id, device)
            
            task_results = []
            
            for limit_steps in STEPS_SWEEP:
                # Train Scratch
                sde_s, head_s, z_s = train_from_scratch_per_task(
                    x_dim, z_dim, full_support[:N_SHOTS], cfg, gen, limit_steps
                )
                
                # Eval Scratch (uses z=0 for fair eval)
                mse_path, mse_head = evaluate_scratch_model(
                    sde_s, head_s, z_s, full_query, cfg, gen
                )
                
                task_results.append({
                    "regime": regime, 
                    "steps_available": limit_steps,
                    "mse_head_scratch": mse_head, 
                    "mse_path_scratch": mse_path
                })
            
            # Incremental save
            df_task = pd.DataFrame(task_results)
            mode = 'a' if os.path.exists(out_file) else 'w'
            header = not os.path.exists(out_file)
            df_task.to_csv(out_file, mode=mode, header=header, index=False)

    print(f"\n✅ Baseline Sweep Complete → {out_file}")
    df = pd.read_csv(out_file)
    print(df.groupby(["regime", "steps_available"])[["mse_head_scratch"]].mean())

if __name__ == "__main__":
    main()


✅ Baseline Sweep Complete → results/scratch_sweep_results.csv
                        mse_head_scratch
regime steps_available                  
testA  20                       0.460999
       40                       0.452975
       50                       0.454167
       80                       0.443618
       100                      0.437832
       120                      0.445297
       201                      0.443703
testB  20                       2.076652
       40                       2.055043
       50                       2.010913
       80                       2.029681
       100                      2.004715
       120                      1.989982
       201                      1.976974
testC  20                       4.020596
       40                       3.944748
       50                       3.951628
       80                       3.874235
       100                      3.853324
       120                      3.897330
       201                      3.874597

In [ ]:
SCRATCH WITH ALL METRICS

In [ ]:
# baselines/adapt_scratch.py
import os
import time
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
# NAIVE REGRESSOR BASELINE (WEAKENED)
ADAPT_STEPS = 50       
ADAPT_LR = 3e-4       # Conservative LR
N_SHOTS = 2           
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]
MC_SAMPLES = 5        # For NLL estimation

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def train_from_scratch_per_task(x_dim, z_dim, support_trajs, config, gen, limit_steps):
    """
    Trains a fresh SDE + Head from random initialization on the support set.
    """
    device = support_trajs.device
    start_time = time.time()
    
    # 1. Initialize Fresh Model (Low Capacity)
    sde = NeuralSDE(x_dim, z_dim, hidden_dim=32).to(device)
    head = ForecastHead(x_dim, z_dim, hidden_dim=32).to(device)

    # 2. Calculate Horizon
    full_T = config.time_grid.T
    full_steps = config.time_grid.n_steps
    dt = full_T / full_steps

    n_sim = limit_steps - 1
    T_train = dt * n_sim
    
    # Slice data
    train_data = support_trajs[:, :limit_steps, :] 
    x0 = train_data[:, 0, :]
    target_final = train_data[:, -1, :]
    
    x_max = config.stability.max_state_abs
    
    # 3. NOISY Z-INIT
    z_init = torch.randn(x0.size(0), z_dim, device=device) * 0.1
    z_init.requires_grad_(True)

    # 4. Optimizer
    optimizer = optim.Adam(
        list(sde.parameters()) + list(head.parameters()) + [z_init], 
        lr=ADAPT_LR, 
        weight_decay=1e-2 
    )

    sde.train(); head.train()

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        
        # Simulate
        traj_pred = simulate_neural_sde_batch(
            sde, x0, z_init, T_train, n_sim, x_max, gen
        )

        pred_final = head(traj_pred[:, -1, :], z_init)
        
        # Loss
        loss = F.mse_loss(pred_final, target_final)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(sde.parameters()) + list(head.parameters()) + [z_init], 1.0
        )
        optimizer.step()

    adapt_time = time.time() - start_time
    return sde, head, adapt_time

def evaluate_scratch_model(sde, head, query_trajs, config, gen):
    """
    Evaluates the task-specific model using MC sampling for NLL.
    """
    sde.eval()
    head.eval()
    device = query_trajs.device
    
    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs
    z_dim = config.latent.latent_dim

    x0_q = query_trajs[:, 0, :]
    # Use z=0 for query because task info is burnt into weights
    z_expanded = torch.zeros(x0_q.size(0), z_dim, device=device)

    mc_preds = []
    with torch.no_grad():
        for _ in range(MC_SAMPLES):
            traj_q = simulate_neural_sde_batch(
                sde, x0_q, z_expanded, T, n_steps, x_max, gen
            )
            mc_preds.append(traj_q)
    
    # Statistics
    mc_tensor = torch.stack(mc_preds, dim=0)
    pred_mean = mc_tensor.mean(dim=0)
    pred_var = mc_tensor.var(dim=0) + 1e-6
    
    mse_rollout = F.mse_loss(pred_mean, query_trajs).item()
    mse_final = F.mse_loss(pred_mean[:, -1], query_trajs[:, -1, :]).item()
    nll = F.gaussian_nll_loss(pred_mean, query_trajs, pred_var).item()

    return mse_rollout, mse_final, nll

def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("💪 Scratch Baseline: FULL METRICS (Time, NLL, Rollout)")
    print("=" * 80)

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    out_file = "results/scratch_sweep_results_full.csv"
    
    # Resume Logic
    completed_keys = set()
    if os.path.exists(out_file):
        print(f"Found existing results. Resuming...")
        try:
            df_exist = pd.read_csv(out_file)
            for _, row in df_exist.iterrows():
                key = f"{row['regime']}_{row['theta_id']}_{row['steps_available']}"
                completed_keys.add(key)
        except:
            print("Error reading CSV, starting fresh.")

    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"\nProcessing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support")
            ds_query = TrajectoryDataset(index_path, regime, "query")
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        
        for theta_id in tqdm(tasks, desc=regime):
            
            # Optimization: Load data only if needed
            needed = False
            for steps in STEPS_SWEEP:
                if f"{regime}_{theta_id}_{steps}" not in completed_keys:
                    needed = True; break
            if not needed: continue

            full_support = get_task_data(ds_supp, theta_id, device)
            full_query = get_task_data(ds_query, theta_id, device)
            
            task_results = []
            
            for limit_steps in STEPS_SWEEP:
                key = f"{regime}_{theta_id}_{limit_steps}"
                if key in completed_keys: continue

                # Train
                sde_s, head_s, adapt_time = train_from_scratch_per_task(
                    x_dim, z_dim, full_support[:N_SHOTS], cfg, gen, limit_steps
                )
                
                # Eval
                mse_rollout, mse_final, nll = evaluate_scratch_model(
                    sde_s, head_s, full_query, cfg, gen
                )
                
                task_results.append({
                    "regime": regime, 
                    "theta_id": theta_id,
                    "steps_available": limit_steps,
                    "mse_rollout": mse_rollout,
                    "mse_final": mse_final,
                    "nll": nll,
                    "adapt_time": adapt_time
                })
            
            if task_results:
                df_task = pd.DataFrame(task_results)
                mode = 'a' if os.path.exists(out_file) else 'w'
                header = not os.path.exists(out_file)
                df_task.to_csv(out_file, mode=mode, header=header, index=False)

    print(f"\n✅ Baseline Sweep Complete → {out_file}")
    if os.path.exists(out_file):
        df = pd.read_csv(out_file)
        # Show full columns
        pd.set_option('display.max_rows', None)
        print(df.groupby(["regime", "steps_available"])[["mse_rollout", "mse_final", "nll", "adapt_time"]].mean())

if __name__ == "__main__":
    main()

In [ ]:
 Baseline Sweep Complete → results/scratch_sweep_results_full.csv
                        mse_rollout  mse_final       nll  adapt_time
regime steps_available                                              
testA  20                  0.218386   0.550099 -0.202622    2.540784
       40                  0.219422   0.550184 -0.218432    4.989502
       50                  0.216989   0.545055 -0.180277    6.120136
       80                  0.225331   0.568268 -0.174954    9.525943
       100                 0.232787   0.586661 -0.158247   11.766371
       120                 0.235538   0.594575 -0.154913   14.083123
       201                 0.252194   0.647596 -0.101518   23.423260
testB  20                  0.727297   2.194223  1.294602    2.470440
       40                  0.732115   2.213722  1.446215    4.982115
       50                  0.723635   2.184714  1.435314    6.102245
       80                  0.716302   2.178968  1.386581    9.445521
       100                 0.730213   2.198539  1.494340   11.735450
       120                 0.725549   2.188204  1.381234   14.016932
       201                 0.755166   2.287436  1.677667   23.284231
testC  20                  1.347368   4.109097  3.340887    2.481991
       40                  1.341208   4.096162  3.290978    4.967978
       50                  1.324923   4.048875  3.227653    6.079834
       80                  1.398792   4.271620  3.378306    9.458902
       100                 1.424309   4.352834  3.814334   11.728956
       120                 1.352780   4.156004  3.456660   14.014084
       201                 1.438717   4.412769  3.883731   23.386274
jovyan@jupyter-ec241027:~/DISSERTATION $ 

--------------------------------------------------------------------------------

ADAPT TRANSFER (WARM STARTING) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

---------------------------------------------------------------------------------

OLD ADAPT TRANSFER 

In [ ]:
# baselines/adapt_transfer_weak.py
# Weak transfer adaptation: load global SDE+Head (no encoder, z=0)
# and fine-tune SDE + Head per task starting from the same z=0.

import os
import copy
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch


ADAPT_STEPS = 50
ADAPT_LR = 1e-2
N_SHOTS = 2


def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    data = [dataset[i][0] for i in rows.index]
    return torch.stack(data).to(device)


def adapt_and_eval_weak(sde_init, head_init, support, query, config, gen):
    """Fine-tune SDE + Head from z=0 global model; evaluate on query."""
    device = support.device
    x_dim = config.basis.x_dim
    z_dim = config.latent.latent_dim

    # 1. Copies for this task
    sde = copy.deepcopy(sde_init)
    head = copy.deepcopy(head_init)

    optimizer = optim.Adam(
        list(sde.parameters()) + list(head.parameters()), lr=ADAPT_LR
    )

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    x0_supp = support[:, 0, :]
    z_zeros_supp = torch.zeros(x0_supp.size(0), z_dim, device=device)
    target_final = support[:, -1, :]

    # 2. Fine-tuning loop on FULL support trajectory
    sde.train()
    head.train()

    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()

        traj = simulate_neural_sde_batch(
            sde, x0_supp, z_zeros_supp, T, n_steps, x_max, gen
        )
        pred_final = head(traj[:, -1, :], z_zeros_supp)

        loss = F.mse_loss(traj, support) + F.mse_loss(pred_final, target_final)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde.parameters(), 1.0)
        optimizer.step()

    # 3. Evaluation on query
    sde.eval()
    head.eval()
    with torch.no_grad():
        x0_q = query[:, 0, :]
        z_zeros_q = torch.zeros(x0_q.size(0), z_dim, device=device)

        traj_q = simulate_neural_sde_batch(
            sde, x0_q, z_zeros_q, T, n_steps, x_max, gen
        )
        pred_q = head(traj_q[:, -1, :], z_zeros_q)

        mse_path = F.mse_loss(traj_q, query).item()
        mse_head = F.mse_loss(pred_q, query[:, -1, :]).item()

    return mse_path, mse_head


def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("🔬 Weak Transfer Baseline: NO ENCODER, z=0, SDE+Head Fine-Tuned Per Task")
    print("=" * 80)
    print(f"Config: {N_SHOTS}-Shot | Steps={ADAPT_STEPS}")
    print("=" * 80 + "\n")

    ckpt_path = "checkpoints/transfer_weak_no_encoder.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"❌ Weak transfer checkpoint not found at {ckpt_path}\n"
            f"Run `python -m baselines.train_transfer_weak` first!"
        )

    ckpt = torch.load(ckpt_path, map_location=device)

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    sde.load_state_dict(ckpt["sde"])
    head.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print("=" * 80)
        print(f"🎯 Processing {regime}")
        print("=" * 80)

        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except RuntimeError:
            print(f"⚠️  Skipping {regime} (Empty dataset)")
            continue

        for theta_id in tqdm(ds_supp.metadata["theta_id"].unique(), desc=regime):
            support = get_task_data(ds_supp, theta_id, device)[:N_SHOTS]
            query = get_task_data(ds_query, theta_id, device)

            mse_path, mse_head = adapt_and_eval_weak(sde, head, support, query, cfg, gen)

            results.append(
                {
                    "regime": regime,
                    "theta_id": theta_id,
                    "n_shots": N_SHOTS,
                    "mse_path_transfer_weak": mse_path,
                    "mse_head_transfer_weak": mse_head,
                }
            )

    df = pd.DataFrame(results)
    os.makedirs("results", exist_ok=True)
    out_path = "results/transfer_weak_results.csv"
    df.to_csv(out_path, index=False)

    print("\n📊 Weak Transfer RESULTS (mean over tasks):")
    print(df.groupby("regime")[["mse_path_transfer_weak", "mse_head_transfer_weak"]].mean())
    print(f"\n💾 Saved to {out_path}")
    print("=" * 80)


if __name__ == "__main__":
    main()


In [ ]:
didnt save output yay

NEW WEAK TRANSFER WITH STEPS
A very basic warm-start: “average SDE + per-task head-only fine-tuning from 2 truncated trajectories.”

The same 20/40/50/…/201 steps_available sweep as meta and scratch.

In [ ]:

# Weak transfer adaptation: load global SDE+Head (no encoder, z=0)
# and fine-tune HEAD ONLY per task starting from the same z=0,
# with a data-efficiency sweep over available support steps.

import os
import copy
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

ADAPT_STEPS = 20         # modest adaptation budget
ADAPT_LR = 1e-2          # for head only
N_SHOTS = 2
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    data = [dataset[i][0] for i in rows.index]
    return torch.stack(data).to(device)

def adapt_and_eval_weak_head_only(sde_init, head_init, support, query, config, gen, limit_steps):
    """
    Fine-tune HEAD ONLY from global SDE+Head with z=0.
    Adaptation uses only the first `limit_steps` of the support trajectories
    and optimizes final-step MSE.
    """
    device = support.device
    x_dim = config.basis.x_dim
    z_dim = config.latent.latent_dim

    # 1. Copies per task
    sde = copy.deepcopy(sde_init).to(device)
    head = copy.deepcopy(head_init).to(device)

    # Freeze SDE, train head only
    sde.eval()
    for p in sde.parameters():
        p.requires_grad = False

    head.train()
    optimizer = optim.Adam(head.parameters(), lr=ADAPT_LR)

    T_full = config.time_grid.T
    n_steps_full = config.time_grid.n_steps
    dt = T_full / n_steps_full
    x_max = config.stability.max_state_abs

    # Slice support to first limit_steps
    support_slice = support[:, :limit_steps, :]
    x0_supp = support_slice[:, 0, :]
    target_final = support_slice[:, -1, :]

    # Number of simulation steps to reach the last observed time
    n_sim = limit_steps - 1
    T_train = dt * n_sim

    z_zeros_supp = torch.zeros(x0_supp.size(0), z_dim, device=device)

    # 2. Fine-tuning loop on truncated support
    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()

        traj = simulate_neural_sde_batch(
            sde, x0_supp, z_zeros_supp, T_train, n_sim, x_max, gen
        )
        pred_final = head(traj[:, -1, :], z_zeros_supp)

        # Final-step MSE only
        loss = F.mse_loss(pred_final, target_final)
        loss.backward()
        optimizer.step()

    # 3. Evaluation on full query trajectories
    sde.eval()
    head.eval()
    with torch.no_grad():
        T = T_full
        n_steps = n_steps_full

        x0_q = query[:, 0, :]
        z_zeros_q = torch.zeros(x0_q.size(0), z_dim, device=device)

        traj_q = simulate_neural_sde_batch(
            sde, x0_q, z_zeros_q, T, n_steps, x_max, gen
        )
        pred_q = head(traj_q[:, -1, :], z_zeros_q)

        mse_path = F.mse_loss(traj_q, query).item()
        mse_head = F.mse_loss(pred_q, query[:, -1, :]).item()

    return mse_path, mse_head

def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("🔬 Weak Transfer Baseline: NO ENCODER, z=0, HEAD-ONLY Fine-Tuning Per Task")
    print("=" * 80)
    print(f"Config: {N_SHOTS}-Shot | Steps={ADAPT_STEPS} | LR={ADAPT_LR}")
    print(f"Steps sweep: {STEPS_SWEEP}")
    print("=" * 80 + "\n")

    ckpt_path = "checkpoints/transfer_weak_no_encoder.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"❌ Weak transfer checkpoint not found at {ckpt_path}\n"
            f"Run `python -m baselines.train_transfer_weak` first!"
        )

    ckpt = torch.load(ckpt_path, map_location=device)

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    sde.load_state_dict(ckpt["sde"])
    head.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print("=" * 80)
        print(f"🎯 Processing {regime}")
        print("=" * 80)

        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except RuntimeError:
            print(f"⚠️  Skipping {regime} (Empty dataset)")
            continue

        for theta_id in tqdm(ds_supp.metadata["theta_id"].unique(), desc=regime):
            support_full = get_task_data(ds_supp, theta_id, device)[:N_SHOTS]
            query = get_task_data(ds_query, theta_id, device)

            for limit_steps in STEPS_SWEEP:
                mse_path, mse_head = adapt_and_eval_weak_head_only(
                    sde, head, support_full, query, cfg, gen, limit_steps
                )

                results.append(
                    {
                        "regime": regime,
                        "theta_id": theta_id,
                        "n_shots": N_SHOTS,
                        "steps_available": limit_steps,
                        "mse_path_transfer_weak": mse_path,
                        "mse_head_transfer_weak": mse_head,
                    }
                )

    df = pd.DataFrame(results)
    os.makedirs("results", exist_ok=True)
    out_path = "results/transfer_weak_results.csv"
    df.to_csv(out_path, index=False)

    print("\n📊 Weak Transfer RESULTS (mean over tasks):")
    print(
        df.groupby(["regime", "steps_available"])[
            ["mse_path_transfer_weak", "mse_head_transfer_weak"]
        ].mean()
    )
    print(f"\n💾 Saved to {out_path}")
    print("=" * 80)

if __name__ == "__main__":
    main()


📊 Weak Transfer RESULTS (mean over tasks):
                        mse_path_transfer_weak  mse_head_transfer_weak
regime steps_available                                                
testA  20                             0.152067                0.403649
       40                             0.152056                0.395191
       50                             0.152064                0.389545
       80                             0.152063                0.370513
       100                            0.152054                0.361802
       120                            0.152045                0.353437
       201                            0.152057                0.334512
testB  20                             0.647731                1.967741
       40                             0.647727                1.887310
       50                             0.647705                1.844065
       80                             0.647696                1.731652
       100                            0.647702                1.662848
       120                            0.647715                1.594155
       201                            0.647705                1.368849
testC  20                             1.283079                3.863816
       40                             1.283096                3.672150
       50                             1.283121                3.572057
       80                             1.283088                3.307008
       100                            1.283096                3.103349
       120                            1.283091                2.895311
       201                            1.283078                2.450659

In [ ]:
WEAK WARM STARTING BUT WITH ALL THE METRICS 

In [ ]:
# baselines/adapt_transfer_weak.py
# Weak transfer adaptation: load global SDE+Head (no encoder, z=0)
# and fine-tune HEAD ONLY per task starting from the same z=0,
# with a data-efficiency sweep over available support steps.
import os
import time
import copy
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# === CONFIG ===
ADAPT_STEPS = 20         # Modest budget for weak baseline
ADAPT_LR = 1e-2          # Head fine-tuning rate
N_SHOTS = 2
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]
MC_SAMPLES = 5           # For NLL estimation

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def adapt_and_eval_weak_head_only(sde_init, head_init, support, query, config, gen, limit_steps):
    """
    Fine-tunes HEAD ONLY with z=0.
    Returns full metrics: adapt_time, nll, mse_rollout, mse_final.
    """
    device = support.device
    x_dim = config.basis.x_dim
    z_dim = config.latent.latent_dim

    # 1. Start Timer
    start_time = time.time()

    # 2. Clone models (Freeze SDE, Train Head)
    sde = copy.deepcopy(sde_init).to(device)
    head = copy.deepcopy(head_init).to(device)

    sde.eval()
    for p in sde.parameters():
        p.requires_grad = False

    head.train()
    optimizer = optim.Adam(head.parameters(), lr=ADAPT_LR)

    # Simulation setup
    T_full = config.time_grid.T
    n_steps_full = config.time_grid.n_steps
    dt = T_full / n_steps_full
    x_max = config.stability.max_state_abs

    # Slice support to limit_steps
    support_slice = support[:, :limit_steps, :]
    x0_supp = support_slice[:, 0, :]
    target_final = support_slice[:, -1, :]
    z_zeros_supp = torch.zeros(x0_supp.size(0), z_dim, device=device)

    n_sim = limit_steps - 1
    T_train = dt * n_sim

    # 3. Fine-tuning Loop
    for _ in range(ADAPT_STEPS):
        optimizer.zero_grad()
        traj = simulate_neural_sde_batch(sde, x0_supp, z_zeros_supp, T_train, n_sim, x_max, gen)
        pred_final = head(traj[:, -1, :], z_zeros_supp)
        loss = F.mse_loss(pred_final, target_final)
        loss.backward()
        optimizer.step()

    adapt_time = time.time() - start_time

    # 4. Evaluation (Monte Carlo for NLL)
    sde.eval()
    head.eval()
    
    x0_q = query[:, 0, :]
    z_zeros_q = torch.zeros(x0_q.size(0), z_dim, device=device)
    
    mc_preds = []
    with torch.no_grad():
        for _ in range(MC_SAMPLES):
            # Full simulation 0 -> T
            traj_q = simulate_neural_sde_batch(sde, x0_q, z_zeros_q, T_full, n_steps_full, x_max, gen)
            mc_preds.append(traj_q)

    # Aggregate MC Samples
    mc_tensor = torch.stack(mc_preds, dim=0) # (MC, B, T, D)
    pred_mean = mc_tensor.mean(dim=0)
    pred_var = mc_tensor.var(dim=0) + 1e-6

    # Metrics
    mse_rollout = F.mse_loss(pred_mean, query).item()
    mse_final = F.mse_loss(pred_mean[:, -1], query[:, -1]).item()
    nll = F.gaussian_nll_loss(pred_mean, query, pred_var).item()

    return {
        "adapt_time": adapt_time,
        "mse_rollout": mse_rollout,
        "mse_final": mse_final,
        "nll": nll
    }

def main():
    device = torch.device(cfg.device)
    print("\n" + "=" * 80)
    print("🔬 Weak Transfer Baseline: FULL METRICS (Time, NLL, Rollout)")
    print("=" * 80)

    # Ensure you have run baselines/train_transfer_weak.py first!
    ckpt_path = "checkpoints/transfer_weak_no_encoder.pt"
    if not os.path.exists(ckpt_path):
        print("⚠️  Warning: Checkpoint not found. Using Random Init (for testing only).")
        # In real usage, raise error or ensure file exists
        ckpt_path = None

    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim
    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    if ckpt_path:
        ckpt = torch.load(ckpt_path, map_location=device)
        sde.load_state_dict(ckpt["sde"])
        head.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)

    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"🎯 Processing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support")
            ds_query = TrajectoryDataset(index_path, regime, "query")
        except: continue

        for theta_id in tqdm(ds_supp.metadata["theta_id"].unique(), desc=regime):
            support_full = get_task_data(ds_supp, theta_id, device)[:N_SHOTS]
            query = get_task_data(ds_query, theta_id, device)

            for limit_steps in STEPS_SWEEP:
                metrics = adapt_and_eval_weak_head_only(
                    sde, head, support_full, query, cfg, gen, limit_steps
                )
                
                metrics["regime"] = regime
                metrics["theta_id"] = theta_id
                metrics["steps_available"] = limit_steps
                results.append(metrics)

    df = pd.DataFrame(results)
    os.makedirs("results", exist_ok=True)
    out_path = "results/transfer_weak_results_full.csv"
    df.to_csv(out_path, index=False)

    print("\n✅ Weak Transfer Full Results:")
    # Print the full table as requested
    pd.set_option('display.max_rows', None)
    print(df.groupby(["regime", "steps_available"])[
        ["mse_rollout", "nll", "adapt_time"]
    ].mean())

if __name__ == "__main__":
    main()

In [ ]:
jovyan@jupyter-ec241027:~/DISSERTATION $ python -c "import pandas as pd; pd.set_option('display.max_rows', None); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000); print(pd.read_csv('results/transfer_weak_results_full.csv').groupby(['regime', 'steps_available'])[['mse_rollout', 'mse_final', 'nll', 'adapt_time']].mean())"
                        mse_rollout  mse_final            nll  adapt_time
regime steps_available                                                   
testA  20                  0.152055   0.403630   46262.049023    0.325918
       40                  0.152056   0.403630   46346.682813    0.551562
       50                  0.152057   0.403626   46329.634310    0.666709
       80                  0.152058   0.403631   46359.630469    1.088742
       100                 0.152057   0.403628   46096.583594    1.314335
       120                 0.152061   0.403639   46291.986979    1.612676
       201                 0.152059   0.403631   46654.936393    2.615140
testB  20                  0.647701   2.015020  191971.619661    0.286535
       40                  0.647703   2.015029  192098.402083    0.540666
       50                  0.647713   2.015051  191380.883073    0.658924
       80                  0.647707   2.015031  192793.431250    1.059950
       100                 0.647703   2.015035  193186.365755    1.290017
       120                 0.647709   2.015049  193162.075911    1.573226
       201                 0.647703   2.015030  193916.877083    2.570878
testC  20                  1.283092   3.985322  380723.768229    0.277785
       40                  1.283094   3.985325  383278.775000    0.517107
       50                  1.283091   3.985336  383553.393750    0.655688
       80                  1.283099   3.985343  385472.097396    1.043302
       100                 1.283095   3.985353  381421.395833    1.241527
       120                 1.283099   3.985328  384328.774479    1.533521
       201                 1.283095   3.985338  381855.146354    2.513951

--------------------------------------------------------------------------------

MAML META 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
minimal first-order MAML-style baseline on your Neural SDE + head, with:

No encoder, no latent conditioning (use 
z
=
0
z=0).

Inner loop: k=5 steps, LR=1e−2, final-step MSE.

Outer loop: meta-train on your existing train regimes.

Adaptation: per-task fine-tuning from the meta-initialization, and a steps_available sweep like the other baselines.

--------------------------------------------------------------------------------

MAML TRAINING 

In [ ]:
# baselines/train_MAML.py
# MAML-style meta-learning baseline on Neural SDE + Head
# No encoder, z = 0. First-order MAML with k inner steps.

import os
import random
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
INNER_STEPS = 5         # k inner-loop adaptation steps
INNER_LR = 1e-2
META_LR = 1e-3
META_BATCH_SIZE = 4     # tasks per meta-batch
EPOCHS = 20             # meta-epochs (you can adjust)
N_SHOTS = 2             # support trajectories per task

def get_task_indices(metadata):
    """Group dataset indices by theta_id."""
    groups = {}
    for idx, row in metadata.iterrows():
        tid = row["theta_id"]
        groups.setdefault(tid, []).append(idx)
    return groups

def sample_task_batch(ds, task_groups, device, n_shots):
    """Sample one task (theta_id) and split into support/query sets."""
    theta_id = random.choice(list(task_groups.keys()))
    idxs = task_groups[theta_id]

    # simplest: first N_SHOTS as support, remaining as query
    # assumes dataset ordering is already support/query-like per theta_id
    if len(idxs) < n_shots + 1:
        return None

    support_idxs = idxs[:n_shots]
    query_idxs = idxs[n_shots:]

    support = torch.stack([ds[i][0] for i in support_idxs]).to(device)
    query = torch.stack([ds[i][0] for i in query_idxs]).to(device)

    return support, query

def inner_adapt(sde, head, support, config, gen):
    """
    Perform k inner-loop gradient steps from the shared init (sde, head)
    on the support set, using final-step MSE and z=0.
    Returns adapted copies (sde_adapted, head_adapted).
    """
    device = support.device
    x_dim = config.basis.x_dim
    z_dim = config.latent.latent_dim

    # Copy parameters for inner-loop
    sde_fast = NeuralSDE(x_dim, z_dim, config.latent.sde_hidden_dim).to(device)
    head_fast = ForecastHead(x_dim, z_dim, config.latent.head_hidden_dim).to(device)
    sde_fast.load_state_dict(sde.state_dict())
    head_fast.load_state_dict(head.state_dict())

    optimizer = optim.SGD(
        list(sde_fast.parameters()) + list(head_fast.parameters()),
        lr=INNER_LR,
    )

    T = config.time_grid.T
    n_steps = config.time_grid.n_steps
    x_max = config.stability.max_state_abs

    x0 = support[:, 0, :]
    target_final = support[:, -1, :]
    z_zeros = torch.zeros(x0.size(0), z_dim, device=device)

    for _ in range(INNER_STEPS):
        optimizer.zero_grad()

        traj = simulate_neural_sde_batch(
            sde_fast, x0, z_zeros, T, n_steps, x_max, gen
        )
        pred_final = head_fast(traj[:, -1, :], z_zeros)

        loss_inner = F.mse_loss(pred_final, target_final)
        loss_inner.backward()
        optimizer.step()

    return sde_fast, head_fast

def main():
    device = torch.device(cfg.device)
    torch.manual_seed(cfg.global_seed if hasattr(cfg, "global_seed") else 12345)
    random.seed(cfg.global_seed if hasattr(cfg, "global_seed") else 12345)

    print("🔥 MAML-style Baseline: Neural SDE + Head, z=0 (no encoder)")
    print(f"Inner steps: {INNER_STEPS}, inner_lr: {INNER_LR}, meta_lr: {META_LR}")
    print(f"Tasks per meta-batch: {META_BATCH_SIZE}, epochs: {EPOCHS}\n")

    # 1. Load meta-train data (use 'train' regime, 'train_inner' split)
    index_path = os.path.join(cfg.paths.data_root, "index.csv")
    train_ds = TrajectoryDataset(index_path, "train", "train_inner", check_shapes=True)
    task_groups = get_task_indices(train_ds.metadata)

    # 2. Initialize model
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim

    sde = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)

    meta_optimizer = optim.Adam(
        list(sde.parameters()) + list(head.parameters()),
        lr=META_LR,
    )

    T = cfg.time_grid.T
    n_steps = cfg.time_grid.n_steps
    x_max = cfg.stability.max_state_abs

    gen = torch.Generator(device=device)
    gen.manual_seed(cfg.global_seed if hasattr(cfg, "global_seed") else 12345)

    # 3. Meta-training loop (first-order MAML)
    for epoch in range(1, EPOCHS + 1):
        meta_loss_sum = 0.0
        num_meta_batches = 0

        # we just loop over a fixed number of meta-batches per epoch
        for _ in tqdm(range(100), desc=f"Meta-epoch {epoch}"):
            meta_optimizer.zero_grad()
            batch_meta_loss = 0.0
            tasks_used = 0

            for _ in range(META_BATCH_SIZE):
                sample = sample_task_batch(train_ds, task_groups, device, N_SHOTS)
                if sample is None:
                    continue
                support, query = sample
                tasks_used += 1

                # Inner adaptation
                sde_fast, head_fast = inner_adapt(sde, head, support, cfg, gen)

                # Compute query loss with adapted params
                x0_q = query[:, 0, :]
                target_final_q = query[:, -1, :]
                z_zeros_q = torch.zeros(x0_q.size(0), z_dim, device=device)

                traj_q = simulate_neural_sde_batch(
                    sde_fast, x0_q, z_zeros_q, T, n_steps, x_max, gen
                )
                pred_final_q = head_fast(traj_q[:, -1, :], z_zeros_q)

                loss_query = F.mse_loss(pred_final_q, target_final_q)
                # First-order MAML: treat sde_fast/head_fast as detached from second-order terms
                # Accumulate meta-loss as sum/mean over tasks
                batch_meta_loss += loss_query

            if tasks_used == 0:
                continue

            batch_meta_loss = batch_meta_loss / tasks_used
            batch_meta_loss.backward()
            meta_optimizer.step()

            meta_loss_sum += batch_meta_loss.item()
            num_meta_batches += 1

        if num_meta_batches > 0:
            print(f"Epoch {epoch} | Meta-loss: {meta_loss_sum / num_meta_batches:.4f}")

    # 4. Save meta-initialization
    os.makedirs("checkpoints", exist_ok=True)
    torch.save(
        {
            "sde": sde.state_dict(),
            "head": head.state_dict(),
        },
        "checkpoints/maml_nsde_init.pt",
    )
    print("\n✅ Saved MAML NSDE initialization to checkpoints/maml_nsde_init.pt")

if __name__ == "__main__":
    main()


training output: 
jovyan@jupyter-ec241027:~/DISSERTATION $ python -m baselines.train_MAML
🔥 MAML-style Baseline: Neural SDE + Head, z=0 (no encoder)
Inner steps: 5, inner_lr: 0.01, meta_lr: 0.001
Tasks per meta-batch: 4, epochs: 20

Meta-epoch 1: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:47<00:00, 11.28s/it]
Epoch 1 | Meta-loss: 0.4533
Meta-epoch 2: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:29<00:00, 11.09s/it]
Epoch 2 | Meta-loss: 0.4395
Meta-epoch 3: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:29<00:00, 11.09s/it]
Epoch 3 | Meta-loss: 0.4355
Meta-epoch 4: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:22<00:00, 11.03s/it]
Epoch 4 | Meta-loss: 0.4505
Meta-epoch 5: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:29<00:00, 11.10s/it]
Epoch 5 | Meta-loss: 0.4514
Meta-epoch 6: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:29<00:00, 11.10s/it]
Epoch 6 | Meta-loss: 0.4422
Meta-epoch 7: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:26<00:00, 11.07s/it]
Epoch 7 | Meta-loss: 0.4420
Meta-epoch 8: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:27<00:00, 11.08s/it]
Epoch 8 | Meta-loss: 0.4429
Meta-epoch 9: 100%|█████████████████████████████████████████████████████████████████████| 100/100 [18:29<00:00, 11.09s/it]
Epoch 9 | Meta-loss: 0.4458
Meta-epoch 10: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:29<00:00, 11.09s/it]
Epoch 10 | Meta-loss: 0.4309
Meta-epoch 11: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:29<00:00, 11.10s/it]
Epoch 11 | Meta-loss: 0.4411
Meta-epoch 12: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:28<00:00, 11.09s/it]
Epoch 12 | Meta-loss: 0.4447
Meta-epoch 13: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:25<00:00, 11.06s/it]
Epoch 13 | Meta-loss: 0.4422
Meta-epoch 14: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:27<00:00, 11.07s/it]
Epoch 14 | Meta-loss: 0.4375
Meta-epoch 15: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:26<00:00, 11.06s/it]
Epoch 15 | Meta-loss: 0.4441
Meta-epoch 16: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:28<00:00, 11.08s/it]
Epoch 16 | Meta-loss: 0.4360
Meta-epoch 17: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:26<00:00, 11.06s/it]
Epoch 17 | Meta-loss: 0.4336
Meta-epoch 18: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:24<00:00, 11.05s/it]
Epoch 18 | Meta-loss: 0.4423
Meta-epoch 19: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:29<00:00, 11.10s/it]
Epoch 19 | Meta-loss: 0.4171
Meta-epoch 20: 100%|████████████████████████████████████████████████████████████████████| 100/100 [18:26<00:00, 11.06s/it]
Epoch 20 | Meta-loss: 0.4355

✅ Saved MAML NSDE initialization to checkpoints/maml_nsde_init.pt
jovyan@jupyter-ec241027:~/DISSERTATION $ 

MAML ADAPT 1

In [ ]:
# baselines/adapt_MAML.py
# Evaluates the MAML-initialized model using the standard efficiency sweep.

import os
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm
from torch.nn import functional as F

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# --- CONFIG ---
INNER_STEPS = 5     
INNER_LR = 0.01     
N_SHOTS = 2
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def adapt_maml_per_task(sde_init, head_init, support, query, config, gen, limit_steps):
    device = support.device
    x_dim = config.basis.x_dim
    z_dim = config.latent.latent_dim

    # 1. Clone Init (Fast Weights)
    sde = copy.deepcopy(sde_init)
    head = copy.deepcopy(head_init)
    
    # 2. Inner Loop (Adaptation)
    # MAML typically uses SGD for the inner loop
    optimizer = optim.SGD(list(sde.parameters()) + list(head.parameters()), lr=INNER_LR)
    
    # Prepare Support Data (Sliced)
    support_slice = support[:, :limit_steps, :]
    x0_s = support_slice[:, 0, :]
    target_s = support_slice[:, -1, :]
    z_zeros = torch.zeros(x0_s.size(0), z_dim, device=device)
    
    T_full = config.time_grid.T
    n_steps_full = config.time_grid.n_steps
    dt = T_full / n_steps_full
    
    # Calculate training horizon
    n_sim = limit_steps - 1
    T_train = dt * n_sim
    x_max = config.stability.max_state_abs

    sde.train()
    head.train()
    
    for _ in range(INNER_STEPS):
        optimizer.zero_grad()
        # Simulate partial trajectory
        traj = simulate_neural_sde_batch(sde, x0_s, z_zeros, T_train, n_sim, x_max, gen)
        pred = head(traj[:, -1, :], z_zeros)
        
        # Loss on final step
        loss = F.mse_loss(pred, target_s)
        loss.backward()
        # Clip inner gradients for safety
        torch.nn.utils.clip_grad_norm_(sde.parameters(), 1.0)
        optimizer.step()
        
    # 3. Evaluation (Full Future)
    sde.eval()
    head.eval()
    with torch.no_grad():
        x0_q = query[:, 0, :]
        z_q = torch.zeros(x0_q.size(0), z_dim, device=device)
        
        # Full simulation for test metric
        traj_q = simulate_neural_sde_batch(sde, x0_q, z_q, T_full, n_steps_full, x_max, gen)
        pred_q = head(traj_q[:, -1, :], z_q)
        
        mse_head = F.mse_loss(pred_q, query[:, -1, :]).item()
        mse_path = F.mse_loss(traj_q, query).item()
        
    return mse_path, mse_head

def main():
    device = torch.device(cfg.device)
    print("🦎 MAML Baseline: Evaluating Data Efficiency")
    
    # CORRECTED PATH HERE
    ckpt_path = "checkpoints/maml_nsde_init.pt"
    
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")
    
    ckpt = torch.load(ckpt_path, map_location=device)
    
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim
    sde_init = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head_init = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    
    # Load Weights
    # Handle different saving formats (dict vs direct state_dict)
    if "sde" in ckpt:
        sde_init.load_state_dict(ckpt["sde"])
        head_init.load_state_dict(ckpt["head"])
    else:
        # Fallback if saved differently
        print("Warning: unexpected checkpoint format, trying best guess load...")
        sde_init.load_state_dict(ckpt["sde"]) 
        head_init.load_state_dict(ckpt["head"])

    gen = torch.Generator(device=device)
    gen.manual_seed(999)
    
    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"🎯 Processing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support", check_shapes=True)
            ds_query = TrajectoryDataset(index_path, regime, "query", check_shapes=True)
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        for theta_id in tqdm(tasks, desc=regime):
            full_support = get_task_data(ds_supp, theta_id, device)
            full_query = get_task_data(ds_query, theta_id, device)
            
            # --- THE SWEEP ---
            for limit_steps in STEPS_SWEEP:
                mse_path, mse_head = adapt_maml_per_task(
                    sde_init, head_init, full_support[:N_SHOTS], full_query, cfg, gen, limit_steps
                )
                
                results.append({
                    "regime": regime,
                    "theta_id": theta_id,
                    "steps_available": limit_steps,
                    "mse_head_maml": mse_head,
                    "mse_path_maml": mse_path
                })
    
    df = pd.DataFrame(results)
    os.makedirs("results", exist_ok=True)
    df.to_csv("results/maml_results.csv", index=False)
    
    print("\n✅ MAML Results:")
    print(df.groupby(["regime", "steps_available"])[["mse_head_maml"]].mean())

if __name__ == "__main__":
    main()

jovyan@jupyter-ec241027:~/DISSERTATION $  python -m baselines.adapt_MAML
🦎 MAML Baseline: Evaluating Data Efficiency
🎯 Processing testA
testA: 100%|██████████████████████████████████████████████████████████████████████| 30/30 [03:29<00:00,  6.98s/it]
🎯 Processing testB
testB: 100%|██████████████████████████████████████████████████████████████████████| 30/30 [03:29<00:00,  6.98s/it]
🎯 Processing testC
testC: 100%|██████████████████████████████████████████████████████████████████████| 30/30 [03:29<00:00,  6.98s/it]

✅ MAML Results:
                        mse_head_maml
regime steps_available               
testA  20                    0.472407
       40                    0.471757
       50                    0.470785
       80                    0.470545
       100                   0.468845
       120                   0.469237
       201                   0.465230
testB  20                    2.099071
       40                    2.096333
       50                    2.091792
       80                    2.087518
       100                   2.083117
       120                   2.077795
       201                   2.057084
testC  20                    4.051485
       40                    4.042599
       50                    4.039126
       80                    4.027762
       100                   4.018524
       120                   4.006778
       201                   3.967517
jovyan@jupyter-ec241027:~/DISSERTATION $ 

In [ ]:
MALM ADAPT WITH ALL EVALUATION METRICS 

In [ ]:
import os
import time
import copy
import pandas as pd
import torch
import torch.optim as optim
from tqdm import tqdm
from torch.nn import functional as F

from config.base_config import cfg
from dataloaders.trajectory_datasets import TrajectoryDataset
from models.neural_sde import NeuralSDE
from models.head import ForecastHead
from training.train_meta import simulate_neural_sde_batch

# === CONFIG ===
INNER_STEPS = 5     # Keep consistent with training
INNER_LR = 0.01     
N_SHOTS = 2
STEPS_SWEEP = [20, 40, 50, 80, 100, 120, 201]
MC_SAMPLES = 5      # Added for NLL

def get_task_data(dataset, theta_id, device):
    rows = dataset.metadata[dataset.metadata["theta_id"] == theta_id]
    idx = rows.index.tolist()
    data = [dataset[i][0] for i in idx]
    return torch.stack(data).to(device)

def adapt_maml_per_task(sde_init, head_init, support, query, config, gen, limit_steps):
    """
    Adapts MAML (Weights) and returns FULL metrics (Time, NLL, Rollout).
    """
    device = support.device
    x_dim = config.basis.x_dim
    z_dim = config.latent.latent_dim

    # 1. Start Timer
    start_time = time.time()

    # 2. Clone Init (Fast Weights)
    sde = copy.deepcopy(sde_init)
    head = copy.deepcopy(head_init)
    
    # 3. Inner Loop (Adaptation)
    optimizer = optim.SGD(list(sde.parameters()) + list(head.parameters()), lr=INNER_LR)
    
    # Prepare Support Data
    support_slice = support[:, :limit_steps, :]
    x0_s = support_slice[:, 0, :]
    target_s = support_slice[:, -1, :]
    z_zeros = torch.zeros(x0_s.size(0), z_dim, device=device)
    
    T_full = config.time_grid.T
    n_steps_full = config.time_grid.n_steps
    dt = T_full / n_steps_full
    n_sim = limit_steps - 1
    T_train = dt * n_sim
    x_max = config.stability.max_state_abs

    sde.train()
    head.train()
    
    for _ in range(INNER_STEPS):
        optimizer.zero_grad()
        traj = simulate_neural_sde_batch(sde, x0_s, z_zeros, T_train, n_sim, x_max, gen)
        pred = head(traj[:, -1, :], z_zeros)
        loss = F.mse_loss(pred, target_s)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(sde.parameters(), 1.0)
        optimizer.step()
        
    adapt_time = time.time() - start_time
    
    # 4. Evaluation (Monte Carlo for NLL)
    sde.eval()
    head.eval()
    
    T_eval = config.time_grid.T
    n_eval = config.time_grid.n_steps
    
    x0_q = query[:, 0, :]
    z_q = torch.zeros(x0_q.size(0), z_dim, device=device)
    
    mc_preds = []
    with torch.no_grad():
        for _ in range(MC_SAMPLES):
            # Full simulation
            traj_q = simulate_neural_sde_batch(sde, x0_q, z_q, T_eval, n_eval, x_max, gen)
            mc_preds.append(traj_q)
            
    # Stack MC samples
    mc_tensor = torch.stack(mc_preds, dim=0)
    pred_mean = mc_tensor.mean(dim=0)
    pred_var = mc_tensor.var(dim=0) + 1e-6
    
    # Metrics
    mse_rollout = F.mse_loss(pred_mean, query).item()
    mse_final = F.mse_loss(pred_mean[:, -1], query[:, -1]).item()
    nll = F.gaussian_nll_loss(pred_mean, query, pred_var).item()
    
    return {
        "adapt_time": adapt_time,
        "mse_rollout": mse_rollout,
        "mse_final": mse_final,
        "nll": nll
    }

def main():
    device = torch.device(cfg.device)
    print("🦎 MAML Baseline: Evaluating with FULL Metrics")
    
    ckpt_path = "checkpoints/maml_nsde_init.pt"
    if not os.path.exists(ckpt_path):
        # Fallback to init if training not done (just to test pipeline)
        print("⚠️ Warning: MAML Checkpoint not found. Using Random Init.")
        # In real run, you MUST have the checkpoint.
        ckpt_path = None 
    
    x_dim = cfg.basis.x_dim
    z_dim = cfg.latent.latent_dim
    sde_init = NeuralSDE(x_dim, z_dim, cfg.latent.sde_hidden_dim).to(device)
    head_init = ForecastHead(x_dim, z_dim, cfg.latent.head_hidden_dim).to(device)
    
    if ckpt_path:
        ckpt = torch.load(ckpt_path, map_location=device)
        if "sde" in ckpt:
            sde_init.load_state_dict(ckpt["sde"])
            head_init.load_state_dict(ckpt["head"])
        else:
            sde_init.load_state_dict(ckpt)

    gen = torch.Generator(device=device)
    gen.manual_seed(999)
    
    results = []
    index_path = os.path.join(cfg.paths.data_root, "index.csv")

    for regime in ["testA", "testB", "testC"]:
        print(f"🎯 Processing {regime}")
        try:
            ds_supp = TrajectoryDataset(index_path, regime, "support")
            ds_query = TrajectoryDataset(index_path, regime, "query")
        except: continue
        
        tasks = ds_supp.metadata["theta_id"].unique()
        for theta_id in tqdm(tasks, desc=regime):
            full_support = get_task_data(ds_supp, theta_id, device)[:N_SHOTS]
            full_query = get_task_data(ds_query, theta_id, device)
            
            for limit_steps in STEPS_SWEEP:
                metrics = adapt_maml_per_task(
                    sde_init, head_init, full_support, full_query, cfg, gen, limit_steps
                )
                
                metrics["regime"] = regime
                metrics["theta_id"] = theta_id
                metrics["steps_available"] = limit_steps
                results.append(metrics)
    
    df = pd.DataFrame(results)
    os.makedirs("results", exist_ok=True)
    df.to_csv("results/maml_results_full.csv", index=False)
    
    print("\n✅ MAML Full Results (Test C):")
    print(df[df['regime']=='testC'].groupby("steps_available")[
        ["mse_rollout", "nll", "adapt_time"]
    ].mean())

if __name__ == "__main__":
    main()

In [ ]:
jovyan@jupyter-ec241027:~/DISSERTATION $ python -c "import pandas as pd; pd.set_option('display.max_rows', None); pd.set_option('display.max_columns', None); pd.set_option('display.width', 1000); print(pd.read_csv('results/maml_results_full.csv').groupby(['regime', 'steps_available'])[['mse_rollout', 'mse_final', 'nll', 'adapt_time']].mean())"
                        mse_rollout  mse_final       nll  adapt_time
regime steps_available                                              
testA  20                  0.234492   0.588850 -0.149184    0.320005
       40                  0.231819   0.581093 -0.134364    0.545839
       50                  0.236070   0.592562 -0.141853    0.666382
       80                  0.234545   0.591086 -0.140704    1.041972
       100                 0.236464   0.592739 -0.154020    1.295516
       120                 0.235662   0.588027 -0.150842    1.548487
       201                 0.236082   0.591949 -0.134138    2.582528
testB  20                  0.762386   2.297340  1.498530    0.281636
       40                  0.767425   2.300999  1.519240    0.548345
       50                  0.764891   2.286320  1.558525    0.673194
       80                  0.763405   2.283472  1.430740    1.050602
       100                 0.764190   2.294427  1.516833    1.296244
       120                 0.766554   2.295546  1.511140    1.545331
       201                 0.758496   2.280669  1.510789    2.583605
testC  20                  1.466324   4.457255  3.721839    0.276325
       40                  1.461511   4.445684  3.822601    0.540632
       50                  1.470563   4.464719  3.804361    0.658070
       80                  1.467138   4.454250  3.800980    1.029242
       100                 1.455224   4.417979  3.624105    1.294130
       120                 1.474676   4.476724  3.936428    1.548518
       201                 1.466904   4.454053  3.710945    2.547397
jovyan@jupyter-ec241027:~/DISSERTATION $ 

In [ ]:
----------------------------------------------------------------------

In [ ]:
GRU METHOD 🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓🤓

In [ ]:
------------------------------------------------------------------------------------

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

df = pd.read_csv("results/gru_baseline_sweep.csv")
print(df.groupby(["regime", "steps_available"])[["MSE_Rollout", "NLL", "Adapt_Time", "MSE_OneStep"]].mean())

                        MSE_Rollout       NLL  Adapt_Time  MSE_OneStep
regime steps_available                                                
testA  20                  0.267013 -1.878389    0.201097     0.002704
       40                  0.245170 -2.400973    0.192967     0.002118
       50                  0.239619 -2.435412    0.190278     0.001957
       80                  0.212226 -2.639800    0.202179     0.001687
       100                 0.185286 -2.716109    0.198577     0.001472
       120                 0.158692 -2.791317    0.204540     0.001384
       201                 0.124761 -2.883841    0.219628     0.001181
testB  20                  1.127097  3.186126    0.179784     0.017146
       40                  0.976746 -0.920243    0.180752     0.010702
       50                  0.806264 -1.428804    0.179368     0.009173
       80                  0.564438 -2.003047    0.186955     0.007869
       100                 0.485215 -2.140487    0.189593     0.007495
      

In [1]:
import pandas as pd

def fix_gru_csv():
    print("🔧 Standardizing GRU Results...")
    try:
        df = pd.read_csv("results/gru_baseline_sweep.csv")
        
        # Rename columns to match your other files
        df = df.rename(columns={
            "MSE_Rollout": "mse_rollout",
            "MSE_OneStep": "mse_final",  # Using OneStep as proxy for final step error
            "NLL": "nll",
            "Adapt_Time": "adapt_time"
        })
        
        df.to_csv("results/gru_baseline_sweep_fixed.csv", index=False)
        print("✅ Saved to results/gru_baseline_sweep_fixed.csv")
        print(df.head())
        
    except FileNotFoundError:
        print("❌ GRU results file not found.")

if __name__ == "__main__":
    fix_gru_csv()

🔧 Standardizing GRU Results...
✅ Saved to results/gru_baseline_sweep_fixed.csv
   adapt_time       nll  mse_final  mse_rollout regime   theta_id  \
0    0.534748 -2.748006   0.001208     0.204655  testA  testA_000   
1    0.213073 -2.943464   0.000946     0.142177  testA  testA_000   
2    0.221985 -3.012148   0.000905     0.110568  testA  testA_000   
3    0.210023 -3.143962   0.000641     0.075385  testA  testA_000   
4    0.221549 -3.189963   0.000571     0.070035  testA  testA_000   

   steps_available  
0               20  
1               40  
2               50  
3               80  
4              100  


In [ ]:
FINAL COMAPRISON 

In [ ]:
================================================================================
🏆 FINAL MSE COMPARISON TABLE (Lower is Better)
================================================================================
Model         Model C (Ours)  GRU Baseline      MAML   Scratch  Weak Transfer
regime steps                                                                 
testA  20           0.163024      0.267013  0.234492  0.218386       0.152055
       40           0.151176      0.245170  0.231819  0.219422       0.152056
       50           0.145006      0.239619  0.236070  0.216989       0.152057
       80           0.143236      0.212226  0.234545  0.225331       0.152058
       100          0.141787      0.185286  0.236464  0.232787       0.152057
       120          0.140469      0.158692  0.235662  0.235538       0.152061
       201          0.143582      0.124761  0.236082  0.252194       0.152059
testB  20           0.615883      1.127097  0.762386  0.727297       0.647701
       40           0.537574      0.976746  0.767425  0.732115       0.647703
       50           0.508573      0.806264  0.764891  0.723635       0.647713
       80           0.546805      0.564438  0.763405  0.716302       0.647707
       100          0.580930      0.485215  0.764190  0.730213       0.647703
       120          0.615714      0.450159  0.766554  0.725549       0.647709
       201          0.687981      0.361534  0.758496  0.755166       0.647703
testC  20           1.164332      2.188238  1.466324  1.347368       1.283092
       40           1.040159      1.559178  1.461511  1.341208       1.283094
       50           1.041468      1.386106  1.470563  1.324923       1.283091
       80           1.188235      1.191122  1.467138  1.398792       1.283099
       100          1.267871      1.042906  1.455224  1.424309       1.283095
       120          1.313557      0.923810  1.474676  1.352780       1.283099
       201          1.358913      0.764135  1.466904  1.438717       1.283095